# Notebook 08: Project Integration

**Time:** 35 minutes  
**Prerequisites:** Notebooks 02-07 complete  
**Goal:** Apply this week's tools to your capstone project and plan your data pipeline

This notebook will:
1. Review what you learned this week
2. Ask Claude to design a domain-specific data pipeline for your project
3. Run a mini pipeline on your project's data
4. Generate a project update document

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 08")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 08


---

## Part 1: Week 3 Review

This week you learned:
- **Web scraping**: trafilatura (traditional) vs Crawl4AI (modern LLM-ready)
- **OCR**: Tesseract (baseline) -> Surya (layout-aware) -> Marker/Docling (2025)
- **ASR**: Whisper, faster-whisper, model size trade-offs
- **Data cleaning**: MinHash dedup, PII removal, DataTrove/FineWeb architecture
- **TTS**: edge-tts (free cloud), Kokoro (local, highest quality)
- **Voice agents**: ASR->LLM->TTS pipeline, Pipecat framework

In [2]:
# Load previous project definition (from HW1/HW2) or create a blank
prev_project = os.path.join(outputs_dir, 'my_project_update.md')
hw2_project = os.path.join(parent_dir, '..', 'Homework2-Submission', 'outputs', 'my_project_update.md')
hw1_project = os.path.join(parent_dir, '..', 'Homework1-Submission', 'outputs', 'my_project_definition.md')

project_context = ""
for path in [hw2_project, hw1_project]:
    if os.path.exists(path):
        with open(path, 'r') as f:
            project_context = f.read()
        print(f"Loaded project context from: {path}")
        print(f"Content preview: {project_context[:300]}...")
        break

if not project_context:
    print("No previous project definition found.")
    print("You'll define your project focus below.")

Loaded project context from: /Users/scottlai/Documents/inferenceAI/Homework3-Submission/../Homework2-Submission/outputs/my_project_update.md
Content preview: # Project Update — Week 2

**Generated:** 2026-04-02 23:31:28
**Student Name:** [Your Name]

---

## Original Project Definition (Week 1)

# My Research Agent Project
**Created:** 2026-03-21 23:43:09


# MY RESEARCH AGENT PROJECT

## 1. PROJECT TITLE
[Your project name]

## 2. THE PROBLEM
[What rese...


---

## Part 2: Data Pipeline Strategy

In [3]:
# TODO 1: Define your project's data strategy

print("=" * 65)
print("TODO 1: Project Data Pipeline Design")
print("=" * 65)
print()

project_description = "[STUDENT: Describe your capstone project in 2-3 sentences]"
project_domain = "[STUDENT: e.g., healthcare, finance, education, legal, etc.]"

strategy_prompt = f"""I'm building this capstone project:
{project_description}

Domain: {project_domain}

Previous context:
{project_context[:500] if project_context else 'No previous context'}

This week I learned about:
- Web scraping (trafilatura, Crawl4AI)
- OCR (Tesseract, Marker, Docling)
- ASR (faster-whisper)
- Data cleaning (MinHash dedup, PII removal, quality filtering)
- TTS (edge-tts, Kokoro)
- Voice agents (Pipecat framework)

Design a data pipeline strategy for my project:
1. What data sources should I collect from? (web, PDFs, audio?)
2. Which extraction tools are best for my domain?
3. What cleaning steps are critical for my use case?
4. Should I incorporate voice capabilities? If so, how?
5. What's the estimated data volume and processing time?

Be specific and practical."""

start = time.time()
response = client.generate(
    prompt=strategy_prompt,
    system="You are a senior ML engineer helping design a data pipeline for a capstone project.",
    max_tokens=700,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo1_reflection = """
[YOUR REFLECTION HERE]

- Which data sources are most relevant for your project?
- Which tools from this week will you actually use in your capstone?
- What's the most challenging data quality issue you expect to face?
"""

print()
print(todo1_reflection)

TODO 1: Project Data Pipeline Design

Response in 16.1s
Model: claude-sonnet-4-6
Tokens: 424 in, 700 out
Stop reason: max_tokens
# Data Pipeline Strategy — Week 2 Capstone Review

---

## ⚠️ Important Notice Before We Begin

Your project definition fields are **all still placeholder text**. I can see:
- Project description: `[STUDENT: Describe your capstone project...]`
- Domain: `[STUDENT: e.g., healthcare, finance...]`
- Problem/Solution/Components: All unfilled template text

**I cannot give you specific, useful pipeline advice without knowing what you're actually building.**

Giving you generic advice here would waste your time and produce a pipeline that may be completely wrong for your use case.

---

## What You Need To Do Right Now

**Fill in your project definition with real content.** Here's a concrete example of what good vs. bad input looks like:

| ❌ What you submitted | ✅ What I need |
|---|---|
| `[Your project name]` | "Legal Brief Summarization Agent" |
| `[What resear

---

## Part 3: Mini Pipeline Demo

In [4]:
# TODO 2: Run a mini data pipeline for your project
#
# Use at least 2 tools from this week to collect and clean
# a small sample of data relevant to your project.

print("=" * 65)
print("TODO 2: Mini Pipeline for Your Project")
print("=" * 65)
print()

# Example: scrape papers + clean them
from src.scraping_utils import scrape_arxiv_abstracts
from src.data_pipeline import run_cleaning_pipeline

# Step 1: Collect data
my_topic = "[STUDENT: Your project's research topic]"
papers = scrape_arxiv_abstracts(topic=my_topic, max_results=5)

# Step 2: Clean the data
raw_texts = [p['abstract'] for p in papers]
pipeline_result = run_cleaning_pipeline(
    texts=raw_texts,
    target_lang="en",
    save_path=os.path.join(outputs_dir, 'my_project_data.json'),
)

print(f"\nCollected and cleaned {len(pipeline_result['cleaned_texts'])} documents for your project.")

todo2_reflection = """
[YOUR REFLECTION HERE]

- What data did you collect and how did you clean it?
- Were any documents removed by the pipeline? Why?
- How would you scale this to a full dataset for your project?
"""

print()
print(todo2_reflection)

TODO 2: Mini Pipeline for Your Project

Scraping arXiv: '[STUDENT: Your project's research topic]' (max 5 papers)

  [1] Topology of minimal surfaces in the sphere from capillarity...
      Authors: Benjy Firester, Raphael Tsiamis
      Abstract: We present a general construction of embedded minimal and constant mean curvature surfaces in $\mathbb{S}^n$ and one-pha...

  [2] Comprehensive List of User Deception Techniques in Emails...
      Authors: Maxime Veit, Mattia Mossano, Tobias Länge...
      Abstract: Email remains a central communication medium, yet its long-standing design and interface conventions continue to enable ...

  [3] Your Pre-trained Diffusion Model Secretly Knows Restoration...
      Authors: Sudarshan Rajagopalan, Vishal M. Patel
      Abstract: Pre-trained diffusion models have enabled significant advancements in All-in-One Restoration (AiOR), offering improved p...

  [4] Weak Solutions to the Bloch Equations with Distant Dipolar Field...
      Authors: Louis-S

---

## Part 4: Generate Project Update

In [5]:
# Generate project update document
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[Not completed]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[Not completed]'

project_update = f"""# Week 3 Project Update: Pretraining Data & Voice Agents

## Project Description

{project_description}

## Data Pipeline Strategy

{response['content'] if 'error' not in response else 'Error generating strategy'}

## Mini Pipeline Results

- Documents collected: {len(raw_texts) if 'raw_texts' in dir() else 'N/A'}
- Documents after cleaning: {len(pipeline_result['cleaned_texts']) if 'pipeline_result' in dir() else 'N/A'}

## Reflections

### Data Strategy
{_todo1}

### Pipeline Execution
{_todo2}

## Tools I Plan to Use

[STUDENT: List the specific tools from Week 3 you'll use in your capstone]

## Next Steps

[STUDENT: What will you build next week with RAG and vector databases?]
"""

update_path = os.path.join(outputs_dir, 'my_project_update.md')
with open(update_path, 'w') as f:
    f.write(project_update)

print(f"Project update saved: {update_path}")

Project update saved: ../outputs/my_project_update.md


In [6]:
# Final reflection
full_reflection = f"""
### Data Pipeline Strategy

{_todo1}

---

### Mini Pipeline Results

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="08",
    section_title="Project Integration",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  424
Total output tokens: 700
Total cost:          $0.0118

Last 1 calls:
  1. [02:26:38] sonnet -- 424in/700out -- $0.0118


---

## Week 3 Complete!

### Submission Checklist

- [ ] All notebooks (00-08) executed with TODOs filled in
- [ ] `outputs/homework_reflection.md` -- your main deliverable (70%)
- [ ] `outputs/my_project_update.md` -- project integration (20%)
- [ ] `outputs/arxiv_papers.json` -- scraped paper data
- [ ] `outputs/cleaned_data.json` -- cleaned pipeline output
- [ ] Audio files in outputs/ -- TTS demonstrations

### What's Next?

**Week 4:** Retrieval-Augmented Generation (RAG) -- combining LLMs with external knowledge, vector databases, and LangChain.